In [1]:
import os
import sys
import json
import argparse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

from glob import glob
from nilearn.glm.second_level import SecondLevelModel
from nilearn.glm.second_level import make_second_level_design_matrix
from nilearn.image import threshold_img
from nilearn.glm import threshold_stats_img

In [2]:
#from atlasreader import create_output

In [3]:
import nilearn
print('nilearn version', nilearn.__version__)

nilearn version 0.12.1


In [4]:
from nilearn import plotting, datasets, surface

# for plotting slices
mni152_template = datasets.load_mni152_template(resolution=1)
# for plotting on surface
fsaverage = datasets.fetch_surf_fsaverage('fsaverage')

[fetch_surf_fsaverage] Dataset found in /ihome/bchandrasekaran/krs228/nilearn_data/fsaverage


In [5]:
task_label = 'badaga'
fwhm = 6.00
space_label = 'MNI152NLin2009cAsym'

# define data directories
bidsroot = os.path.join('/bgfs/bchandrasekaran/krs228/data/', 
                        'SSP/',
                        'data_bids',
                       )
nilearn_dir = os.path.join(bidsroot, 'derivatives', 'nilearn')

# Group-level analyses

Based on [nilearn documentation](https://nilearn.github.io/stable/auto_examples/05_glm_second_level/plot_thresholding.html#statistical-testing-of-a-second-level-analysis)

### Build the group-level design matrix

#### Read the `participants.tsv` file from the BIDS root directory

In [ ]:
participants_fpath = os.path.join(bidsroot, 'participants.tsv')
participants_df = pd.read_csv(participants_fpath, sep='\t')

# subjects to ignore (not fully processed, etc.)
ignore_subs = ['sub-SSP001', 'sub-SSP002',
               'sub-SSP005', 'sub-SSP012',
               'sub-SSP014', 
               'sub-SSP058', 
               'sub-SSP069', 'sub-SSP072', 
               'sub-SSP076', # missing MRI files
               'sub-SSP102', # participant left after a few minutes
               'sub-SSP106', 'sub-SSP107', # TODO(confirm with team): excluded here with no recorded reason, but first-level GLMs ARE computed for them in univariate_first-level.ipynb -- verify this exclusion is intentional before publication
               ]
participants_df.drop(participants_df[participants_df.participant_id.isin(ignore_subs)].index, inplace=True)

# re-sort by participant ID
participants_df.sort_values(by=['participant_id'], ignore_index=True, inplace=True)

In [7]:
participants_df.head(10)

,participant_id,sex,group,age,toni_index,ctopp_phon_awareness,celf_core_language,celf_receptive_language,celf_expressive_language,pta,win_correct_avg,win_score_avg
0,sub-SSP008,M,control,8.550685,104.0,98.0,113.0,117.0,120.0,NaN,NaN,NaN
1,sub-SSP009,M,control,9.879452,101.0,98.0,111.0,89.0,116.0,NaN,NaN,NaN
2,sub-SSP011,M,control,11.495890,114.0,84.0,117.0,105.0,122.0,NaN,NaN,NaN
3,sub-SSP013,M,control,11.027397,99.0,71.0,85.0,77.0,89.0,NaN,NaN,NaN
4,sub-SSP015,M,control,9.690411,100.0,100.0,96.0,107.0,98.0,NaN,NaN,NaN
5,sub-SSP017,M,control,11.219178,115.0,96.0,117.0,113.0,118.0,NaN,NaN,NaN
6,sub-SSP018,F,cws,8.969863,102.0,103.0,102.0,94.0,106.0,NaN,NaN,NaN
7,sub-SSP020,F,cws,13.238356,98.0,110.0,133.0,122.0,135.0,NaN,NaN,NaN
8,sub-SSP021,M,cws,8.367123,115.0,129.0,133.0,129.0,134.0,NaN,NaN,NaN
9,sub-SSP028,M,control,10.819178,107.0,116.0,100.0,102.0,100.0,NaN,26.5,4.8


In [ ]:
# create group-specific lists of subject IDs
# group labels in participants.tsv have been observed with inconsistent casing (e.g. 'CWS' vs 'cws'),
# so compare against a case- and whitespace-normalized copy rather than the raw strings
group_norm = participants_df.group.str.strip().str.lower()
sub_list_cwns = list(participants_df.participant_id[group_norm == 'control'])
sub_list_cws = list(participants_df.participant_id[group_norm == 'cws'])

# full subject list used by every contrast section below (sorted for a deterministic file order)
subjects_label = sorted(sub_list_cwns + sub_list_cws)

#### Create confounds dataframe

**Modeling choices (recorded for reproducibility):** continuous covariates below are **mean-centered** (`x - x.mean()`), not divided by their mean and not z-scored -- this keeps covariate=0 at the sample average (so group main-effect contrasts are evaluated "at an average participant") while leaving contrast magnitudes in the covariates' original units. **Missing covariate values** are handled per-contrast, per-subject (`prepare_group_inputs`, further down): a subject is dropped only from the specific contrast's design matrix if they're missing a covariate that contrast actually uses, and the drop is logged -- not the previous behavior of silently dropping an entire covariate for every subject if even one subject was missing that value.

In [9]:
participants_df.head()

,participant_id,sex,group,age,toni_index,ctopp_phon_awareness,celf_core_language,celf_receptive_language,celf_expressive_language,pta,win_correct_avg,win_score_avg
0,sub-SSP008,M,control,8.550685,104.0,98.0,113.0,117.0,120.0,NaN,NaN,NaN
1,sub-SSP009,M,control,9.879452,101.0,98.0,111.0,89.0,116.0,NaN,NaN,NaN
2,sub-SSP011,M,control,11.495890,114.0,84.0,117.0,105.0,122.0,NaN,NaN,NaN
3,sub-SSP013,M,control,11.027397,99.0,71.0,85.0,77.0,89.0,NaN,NaN,NaN
4,sub-SSP015,M,control,9.690411,100.0,100.0,96.0,107.0,98.0,NaN,NaN,NaN


In [10]:
cov_design_df = participants_df.copy()

In [11]:
cov_design_df.sex = np.where(cov_design_df['sex'] == 'F', 1, 0)

In [12]:
cov_design_df.columns

Index(['participant_id', 'sex', 'group', 'age', 'toni_index',
       'ctopp_phon_awareness', 'celf_core_language', 'celf_receptive_language',
       'celf_expressive_language', 'pta', 'win_correct_avg', 'win_score_avg'],
      dtype='object')

In [ ]:
cols = [
    'age',
    'toni_index',
    'ctopp_phon_awareness',
    'celf_core_language',
    'celf_receptive_language',
    'celf_expressive_language',
    'pta',
    'win_correct_avg',
    'win_score_avg'
]

# mean-center (not divide-by-mean): keeps covariate=0 at the sample average, so group
# main-effect contrasts (e.g. [1,0,0,0]) are evaluated "at an average participant"
# rather than extrapolated to an out-of-range point.
cov_design_df[cols] = cov_design_df[cols].apply(lambda x: x - x.mean())

In [ ]:
# use the same case-/whitespace-normalized comparison as sub_list_cwns/sub_list_cws above,
# so the one-hot columns and the file-list subject groupings can never disagree
cov_group_norm = cov_design_df['group'].str.strip().str.lower()
cov_design_df['CWS']  = np.where(cov_group_norm == 'cws', 1, 0)
cov_design_df['CWNS'] = np.where(cov_group_norm == 'control', 1, 0)

In [ ]:
# Keep participant_id as the index (rather than dropping it) so covariate rows can always be
# explicitly aligned to a given contrast's subject list by ID -- see prepare_group_inputs()
# below, which reindexes by subject_ids instead of relying on cov_design_df and l1_fnames
# happening to share the same sort order.
cov_design_df = cov_design_df.set_index('participant_id').drop(columns=['group'])

In [16]:
cols_to_front = ['CWS', 'CWNS']
cov_design_df = cov_design_df[cols_to_front + [c for c in cov_design_df.columns if c not in cols_to_front]]

In [17]:
#design_df = cov_design_df[['CWS', 'CWNS', 'age', 'sex']]

In [18]:
len(cov_design_df.columns)

12

In [ ]:
# NOTE: previously this cell did `cov_design_df = cov_design_df.dropna(axis='columns')`,
# which silently dropped an ENTIRE covariate for every subject if even one subject was
# missing that value. Instead, report what's missing here; NaN handling happens per-contrast
# in prepare_group_inputs() below, dropping only the subjects actually missing a column used
# in that specific contrast's design matrix.
missing_report = cov_design_df.isna().sum()
missing_report = missing_report[missing_report > 0]
if len(missing_report) > 0:
    print("Covariates with missing values (subjects will be dropped per-contrast, "
          "only for columns actually used in that contrast's design matrix):")
    print(missing_report)
else:
    print("No missing values in cov_design_df.")


def prepare_group_inputs(l1_dir, task_label, contrast_label, subject_ids, cov_design_df, columns):
    """Get each subject's first-level effect-size map for `contrast_label`, and the matching
    covariate design-matrix subset (`columns`) -- guaranteed to be in the same subject order,
    with the same subjects dropped from both if a file is missing or a covariate is NaN, so
    the images fed to SecondLevelModel.fit() and the design matrix rows can never silently
    drift out of alignment (previously both relied on cov_design_df and l1_fnames happening
    to share the same participant_id sort order).

    Not every subject has every contrast file -- a run may have been excluded for excessive
    motion, first-level may not have finished for everyone, a condition may not have been
    estimable, etc. Subjects missing a file for THIS contrast are dropped from THIS contrast
    only (logged below), the same way subjects missing a covariate are dropped -- not raised
    as an error, since that would kill the whole contrast section over a handful of subjects.
    """
    l1_fnames = []
    missing_files = []
    for sub_id in subject_ids:
        pattern = (l1_dir + f'/{sub_id}/{sub_id}_task-{task_label}_fwhm-6_'
                   f'contrast-{contrast_label}_stat-effect_statmap.nii.gz')
        matches = sorted(glob(pattern))
        if len(matches) == 0:
            missing_files.append(sub_id)
            l1_fnames.append(None)
        else:
            l1_fnames.append(matches[0])

    design_df = cov_design_df.loc[subject_ids, columns]
    missing_file_mask = pd.Series([f is None for f in l1_fnames], index=design_df.index)
    na_covariate_mask = design_df.isna().any(axis=1)
    drop_mask = missing_file_mask | na_covariate_mask

    if missing_file_mask.any():
        print(f"Dropping {missing_file_mask.sum()} subject(s) with no contrast-{contrast_label} "
              "effect map on disk (not every subject has every contrast -- e.g. a run excluded "
              "for motion, or first-level not yet finished for them): "
              f"{list(design_df.index[missing_file_mask])}")
    if na_covariate_mask.any():
        print(f"Dropping {na_covariate_mask.sum()} subject(s) missing a covariate used in the "
              f"contrast-{contrast_label} design matrix:")
        print(design_df.loc[na_covariate_mask])

    if drop_mask.all():
        raise RuntimeError(
            f"No usable subjects remain for contrast-{contrast_label} after dropping those "
            "missing a file or a covariate -- nothing to fit."
        )

    keep = ~drop_mask.values
    l1_fnames_retained = [f for f, k in zip(l1_fnames, keep) if k]
    design_df_retained = design_df.loc[~drop_mask]
    return l1_fnames_retained, design_df_retained

In [35]:
cov_design_df

,CWS,CWNS,sex,age,toni_index,ctopp_phon_awareness,celf_core_language,celf_receptive_language,celf_expressive_language
0,0,1,0,0.867778,0.990681,0.970924,1.020817,1.078341,1.071013
1,0,1,0,1.002629,0.962104,0.970924,1.002749,0.820276,1.035312
2,0,1,0,1.166676,1.085939,0.832221,1.056952,0.967742,1.088863
3,0,1,0,1.119130,0.943052,0.703425,0.767871,0.709677,0.794334
4,0,1,0,0.983444,0.952578,0.990739,0.867243,0.986175,0.874660
5,0,1,0,1.138593,1.095465,0.951109,1.056952,1.041475,1.053163
6,0,0,1,0.910318,0.971630,1.020461,0.921445,0.866359,0.946061
7,0,0,1,1.343512,0.933527,1.089813,1.201493,1.124424,1.204889
8,0,0,0,0.849149,1.095465,1.278053,1.201493,1.188940,1.195964
9,0,1,0,1.097999,1.019259,1.149257,0.903378,0.940092,0.892511


## Group-level motion covariate and shared per-contrast helpers

In [ ]:
# First-level now writes one motion_qc.csv per subject (sub-*/sub-*_motion_qc.csv) instead of a
# single shared file, to avoid a race when many subjects' first-level jobs run in parallel on the
# cluster and all append to the same path. Glob + concatenate them all here. If none exist yet
# (e.g. first-level hasn't been re-run with the updated script/notebook), fall back to no motion
# covariate rather than erroring, and say so explicitly.
motion_qc_fpaths = sorted(glob(os.path.join(nilearn_dir, 'run-all_contrast-snr', 'sub-*', 'sub-*_motion_qc.csv')))
has_motion_covariate = False
if len(motion_qc_fpaths) > 0:
    motion_qc_df = pd.concat([pd.read_csv(f) for f in motion_qc_fpaths], ignore_index=True)
    motion_qc_df = motion_qc_df.set_index('subject_id')
    cov_design_df['mean_fd'] = cov_design_df.index.map(motion_qc_df['mean_fd'])
    if cov_design_df['mean_fd'].notna().all():
        has_motion_covariate = True
    else:
        print('WARNING: motion_qc.csv is missing entries for some subjects; '
              'not including mean_fd as a covariate.')
        cov_design_df = cov_design_df.drop(columns=['mean_fd'])
else:
    print(f'NOTE: no sub-*_motion_qc.csv found under {os.path.join(nilearn_dir, "run-all_contrast-snr")} '
          '(re-run first-level with the updated script/notebook to generate them) -- proceeding '
          'without a group-level motion covariate.')

design_columns = ['CWS', 'CWNS', 'age', 'sex'] + (['mean_fd'] if has_motion_covariate else [])
print('Group-level design matrix columns:', design_columns)

**Multiple-comparisons note (deliberate choice):** every contrast below is FDR cluster-corrected independently (`alpha=0.05`, no correction *across* the 9 contrasts). This is standard practice in task-fMRI when each contrast is treated as an a priori hypothesis, but it means the family-wise error rate across all reported maps is higher than 0.05 -- state this explicitly in any writeup. (Previously this was also *inconsistent*: every within-group map used FDR correction except the one existing group-difference contrast, which was left uncorrected -- every map below, including every group-difference contrast, now uses the same FDR correction.)

In [ ]:
ALPHA = 0.05
CLUSTER_THRESHOLD = 10
HEIGHT_CONTROL = 'fdr'

group_out_dir = os.path.join(nilearn_dir, 'group_run-all')
if not os.path.exists(group_out_dir):
    os.makedirs(group_out_dir)


def contrast_vector(design_df, positive_col, negative_col=None):
    """Build a second-level contrast array by column name instead of a hardcoded positional
    list like [1, -1, 0, 0] -- avoids a contrast silently pointing at the wrong column whenever
    a covariate (e.g. mean_fd) is added to or removed from design_df.
    """
    vec = np.zeros(len(design_df.columns))
    vec[design_df.columns.get_loc(positive_col)] = 1
    if negative_col is not None:
        vec[design_df.columns.get_loc(negative_col)] = -1
    return vec


def compute_group_contrast(second_level_model, second_level_contrast,
                           alpha=ALPHA, cluster_threshold=CLUSTER_THRESHOLD,
                           height_control=HEIGHT_CONTROL):
    """Compute + threshold one contrast from an already-fit SecondLevelModel. FDR cluster
    correction is the consistent default for every group-level map in this notebook (see the
    multiple-comparisons note above).
    """
    z_map = second_level_model.compute_contrast(second_level_contrast=second_level_contrast,
                                                output_type='z_score')
    thresholded_map, zthresh = threshold_stats_img(
        z_map, alpha=alpha, height_control=height_control,
        cluster_threshold=cluster_threshold, two_sided=True,
    )
    return z_map, thresholded_map, zthresh


def plot_mosaic_with_contours(z_map, thresholded_map, zthresh, alpha, title, out_fpath=None):
    display = plotting.plot_stat_map(
        z_map,
        bg_img=mni152_template,
        transparency=z_map, transparency_range=[1, zthresh],
        black_bg=False,
        display_mode='mosaic',
        cmap='seismic',
        title=title,
    )
    display.add_contours(
        thresholded_map, filled=False, levels=[-zthresh, zthresh], colors=["k", "k"]
    )
    print('FDR alpha=%.3g cluster-corrected (>%d voxels) threshold z=%.3g'
         % (alpha, CLUSTER_THRESHOLD, zthresh))
    if out_fpath is not None:
        display.savefig(out_fpath, dpi=1000)
    return display

*(The exploratory "per-covariate effect" loop that used to live here, testing each individual covariate's own coefficient for the `qMinusN6` contrast only, was removed: it only ever covered one of the 9 contrasts, and would have broken under the new per-contrast NaN handling above since it used the full, unfiltered `cov_design_df` directly. It can be rebuilt from `prepare_group_inputs`/`compute_group_contrast` above if still wanted.)*

## contrast-qMinusN6

### Get files, fit model

In [ ]:
contrast_label = 'qMinusN6'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-qMinusN6, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-qMinusN6_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-qMinusN6, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-qMinusN6_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-qMinusN6, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-qMinusN6_view-mosaic.png'),
)

## contrast-q

### Get files, fit model

In [ ]:
contrast_label = 'q'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-q, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-q_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-q, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-q_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-q, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-q_view-mosaic.png'),
)

## contrast-8

### Get files, fit model

In [ ]:
contrast_label = '8'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-8, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-8_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-8, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-8_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-8, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-8_view-mosaic.png'),
)

## contrast-0

### Get files, fit model

In [ ]:
contrast_label = '0'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-0, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-0_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-0, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-0_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-0, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-0_view-mosaic.png'),
)

## contrast-n2

### Get files, fit model

In [ ]:
contrast_label = 'n2'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-n2, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-n2_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-n2, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-n2_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-n2, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-n2_view-mosaic.png'),
)

## contrast-n6

### Get files, fit model

In [ ]:
contrast_label = 'n6'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-n6, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-n6_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-n6, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-n6_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-n6, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-n6_view-mosaic.png'),
)

## contrast-qMinus0

### Get files, fit model

In [ ]:
contrast_label = 'qMinus0'
l1_dir = os.path.join(nilearn_dir, 'run-all_contrast-snr')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-qMinus0, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-qMinus0_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-qMinus0, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-qMinus0_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-qMinus0, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-qMinus0_view-mosaic.png'),
)

## contrast-sound

### Get files, fit model

In [ ]:
contrast_label = 'sound'
l1_dir = os.path.join(nilearn_dir, 'run-all')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-sound, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-sound_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-sound, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-sound_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-sound, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-sound_view-mosaic.png'),
)

## contrast-response

### Get files, fit model

In [ ]:
contrast_label = 'response'
l1_dir = os.path.join(nilearn_dir, 'run-all')
print(l1_dir)

l1_fnames, design_df = prepare_group_inputs(l1_dir, task_label, contrast_label, subjects_label,
                                            cov_design_df, design_columns)
second_level_model = SecondLevelModel().fit(l1_fnames, design_matrix=design_df)

n_cws = int(design_df['CWS'].sum())
n_cwns = int(design_df['CWNS'].sum())
corrected_for = 'age/sex' + ('/motion' if has_motion_covariate else '')
print(f'n_cws={n_cws}, n_cwns={n_cwns}, corrected for: {corrected_for}')

### CWNS speaking group

In [ ]:
z_map_cwns, thresh_cwns, zthresh_cwns = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWNS'))

plot_mosaic_with_contours(
    z_map_cwns, thresh_cwns, zthresh_cwns, ALPHA,
    title=(f'CWNS group (n={n_cwns}) contrast-response, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cwns_contrast-response_view-mosaic.png'),
)

### CWS speaking group

In [ ]:
z_map_cws, thresh_cws, zthresh_cws = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS'))

plot_mosaic_with_contours(
    z_map_cws, thresh_cws, zthresh_cws, ALPHA,
    title=(f'CWS group (n={n_cws}) contrast-response, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-cws_contrast-response_view-mosaic.png'),
)

### Group differences (CWS vs. CWNS)

In [ ]:
z_map_diff, thresh_diff, zthresh_diff = compute_group_contrast(
    second_level_model, contrast_vector(design_df, 'CWS', 'CWNS'))

plot_mosaic_with_contours(
    z_map_diff, thresh_diff, zthresh_diff, ALPHA,
    title=(f'CWS (n={n_cws}) vs. CWNS (n={n_cwns}) group differences, '
          f'contrast-response, {corrected_for}-corrected'),
    out_fpath=os.path.join(group_out_dir, 'group-diff_contrast-response_view-mosaic.png'),
)